# ROC Curve with Logistic Regression — Arrow Binary Classification

**Course:** IRS 6to — Machine Learning, Week 2  
**Task:** Classify images as LEFT arrow (class 0) or RIGHT arrow (class 1)  
**Method:** HOG feature extraction → Logistic Regression → ROC / AUC analysis

---

## What This Notebook Covers

| Step | Goal |
|------|------|
| 1 | Load images, crop to bounding box, convert to grayscale |
| 2 | Extract HOG features and visualize them |
| 3 | Build the feature matrix X and label vector y |
| 4 | Train Logistic Regression |
| 5 | Get probability scores on the test set |
| 6 | Compute and plot the ROC curve + AUC |
| 7 | Select best threshold using the Youden Index |

---

## Background: Why This Pipeline?

### The Problem with Raw Images
Logistic regression expects a **flat vector of numbers**, not a 2D image.  
We need to convert each image into a compact, meaningful numerical representation.

### Why Crop First?
This dataset is YOLO-formatted: each image has a `.txt` label file with a bounding box telling us exactly where the arrow is.  
**74% of images** have the arrow covering less than 30% of the frame — the rest is background noise (hands, walls, ceilings).  
Cropping to the bounding box ensures HOG describes only the arrow shape.

### Why HOG?
HOG (Histogram of Oriented Gradients) captures **edge directions** in local image regions.  
Arrows are defined by their shape — HOG is ideal for this. It produces a compact ~8,100-value vector per image.

### Why ROC/AUC and not just Accuracy?
- **Accuracy** depends on a fixed threshold (usually 0.5) and is misleading with imbalanced classes.
- **ROC/AUC** evaluates the model's discriminative ability across **all possible thresholds**, giving a threshold-independent measure of quality.
- **AUC = 1.0** → perfect; **AUC = 0.5** → no better than random.

## Setup — Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import joblib

from PIL import Image
from skimage.feature import hog
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_curve, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)

%matplotlib inline

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.getcwd()
ARCHIVE_DIR = os.path.join(BASE_DIR, "archive")

CLASS_NAMES = {0: "LEFT", 1: "RIGHT"}
IMG_SIZE    = (128, 128)

HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=False,
    channel_axis=None,
)

print("All imports OK.")
print(f"Working directory : {BASE_DIR}")
print(f"Archive directory : {ARCHIVE_DIR}")

## Shared Utilities

These two functions are used throughout every step of the pipeline.  
Defined once here so all cells below can call them directly.

In [ ]:
def read_label_file(img_filename, label_dir):
    """
    Read a YOLO-format label file.

    YOLO format (one line per object):
        <class_id>  <cx>  <cy>  <bw>  <bh>
    All coordinates are normalized to [0, 1] relative to image size.

    Returns (class_id, cx, cy, bw, bh) or None if the file is missing/empty.
    """
    stem = os.path.splitext(img_filename)[0]
    path = os.path.join(label_dir, stem + ".txt")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        line = f.readline().strip()
    if not line:
        return None
    parts = line.split()
    return int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])


def preprocess(img_path, label_info):
    """
    Full preprocessing pipeline for one image:
      1. Open as RGB
      2. Crop to bounding box (using normalized YOLO coords)
      3. Convert crop to grayscale
      4. Resize to IMG_SIZE (128x128)
      5. Normalize pixel values to [0, 1]

    Returns a float32 numpy array of shape (128, 128).
    """
    _, cx, cy, bw, bh = label_info
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    # Convert normalized bbox to pixel coordinates
    x1 = max(0, int((cx - bw / 2) * w))
    y1 = max(0, int((cy - bh / 2) * h))
    x2 = min(w, int((cx + bw / 2) * w))
    y2 = min(h, int((cy + bh / 2) * h))

    crop = img.crop((x1, y1, x2, y2)).convert("L")  # grayscale crop
    crop = crop.resize(IMG_SIZE, Image.BILINEAR)
    return np.array(crop, dtype=np.float32) / 255.0


print("Utility functions defined.")

---
## Step 1 — Dataset Exploration & Image Preprocessing

### What We Do
1. Count images and check class balance in each split
2. Look at raw images with their bounding boxes overlaid
3. Apply the crop → grayscale → resize pipeline and verify the result visually

### Why Bounding Box Crop?

Each YOLO label file encodes the arrow location as:  
```
class_id   cx    cy    width  height   (all normalized 0-1)
    0      0.57  0.33  0.86   0.66
```
We convert these to pixel coordinates and crop before doing anything else.  
This removes irrelevant background so HOG only describes the arrow shape.

In [ ]:
# ── Count images per split ────────────────────────────────────────────────────
splits = ["train", "test", "validate"]

for split in splits:
    img_dir = os.path.join(ARCHIVE_DIR, "images", split)
    lbl_dir = os.path.join(ARCHIVE_DIR, "labels", split)
    counts = {0: 0, 1: 0}
    for fname in os.listdir(img_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        info = read_label_file(fname, lbl_dir)
        if info:
            counts[info[0]] = counts.get(info[0], 0) + 1
    total = sum(counts.values())
    print(f"{split:>10}  |  LEFT={counts[0]:>4}  RIGHT={counts[1]:>4}  total={total:>4}")

In [ ]:
# ── How much of each image does the arrow actually cover? ─────────────────────
lbl_dir = os.path.join(ARCHIVE_DIR, "labels", "train")
areas = []
for fname in os.listdir(lbl_dir):
    path = os.path.join(lbl_dir, fname)
    with open(path) as f:
        line = f.readline().strip().split()
    if len(line) >= 5:
        areas.append(float(line[3]) * float(line[4]))  # bw * bh

areas = np.array(areas)
print(f"Arrow bbox area (as fraction of image):")
print(f"  min={areas.min():.3f}  mean={areas.mean():.3f}  max={areas.max():.3f}")
print(f"  Images where arrow < 30% of frame : {(areas < 0.3).mean()*100:.1f}%")
print(f"  → Most images have significant background that would confuse HOG.")
print(f"  → Cropping to the bounding box is essential.")

In [ ]:
# ── Visual: raw image + bounding box ─────────────────────────────────────────
img_dir = os.path.join(ARCHIVE_DIR, "images", "train")
lbl_dir = os.path.join(ARCHIVE_DIR, "labels", "train")

# Find 4 images where the arrow is small (lots of background)
small_samples = []
for fname in sorted(os.listdir(img_dir)):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    info = read_label_file(fname, lbl_dir)
    if info and info[3] * info[4] < 0.15:
        small_samples.append((fname, info))
    if len(small_samples) == 4:
        break

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Raw image vs Cropped to bounding box", fontsize=13, fontweight="bold")

for col, (fname, info) in enumerate(small_samples):
    cls, cx, cy, bw, bh = info
    img = Image.open(os.path.join(img_dir, fname)).convert("RGB")
    w, h = img.size

    # Top row: full image with red bbox
    axes[0, col].imshow(img)
    x1 = (cx - bw/2)*w;  y1 = (cy - bh/2)*h
    rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                               linewidth=2, edgecolor="red", facecolor="none")
    axes[0, col].add_patch(rect)
    axes[0, col].set_title(f"Full — {CLASS_NAMES[cls]}\nArrow={bw*bh*100:.0f}% of image", fontsize=8)
    axes[0, col].axis("off")

    # Bottom row: cropped
    x1p = max(0, int((cx - bw/2)*w));  y1p = max(0, int((cy - bh/2)*h))
    x2p = min(w, int((cx + bw/2)*w));  y2p = min(h, int((cy + bh/2)*h))
    crop = img.crop((x1p, y1p, x2p, y2p))
    axes[1, col].imshow(crop)
    axes[1, col].set_title(f"Cropped\n{x2p-x1p}×{y2p-y1p}px", fontsize=8)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ── Visual: final preprocessed crops (grayscale 128x128) ─────────────────────
img_dir = os.path.join(ARCHIVE_DIR, "images", "train")
lbl_dir = os.path.join(ARCHIVE_DIR, "labels", "train")

all_samples = []
for fname in sorted(os.listdir(img_dir)):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    info = read_label_file(fname, lbl_dir)
    if info:
        all_samples.append((fname, info))

left_samples  = [(f, i) for f, i in all_samples if i[0] == 0][:5]
right_samples = [(f, i) for f, i in all_samples if i[0] == 1][:5]

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Step 1 — Preprocessed crops: 128×128 grayscale", fontsize=12, fontweight="bold")

for ax, (fname, info) in zip(axes.flat, left_samples + right_samples):
    arr = preprocess(os.path.join(img_dir, fname), info)
    ax.imshow(arr, cmap="gray")
    ax.set_title(f"{CLASS_NAMES[info[0]]} | bbox {info[3]*info[4]*100:.0f}%\n{fname}", fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.show()
print("First row = LEFT (class 0), second row = RIGHT (class 1)")

---
## Step 2 — HOG Feature Extraction

### What is HOG?

HOG (Histogram of Oriented Gradients) transforms an image into a compact vector that describes **where edges are and what direction they point**.

**Algorithm:**
1. Compute pixel-level gradients (intensity changes in X and Y direction)
2. Divide the image into small **cells** (8×8 pixels each)
3. In each cell, build a **histogram of gradient directions** (9 bins, 0°–180°)
4. Normalize across **blocks** (2×2 cells) for lighting robustness
5. Concatenate all histograms → **one flat feature vector**

**Why it works for arrows:**  
A LEFT arrow has its dominant edges pointing left (the tip) and diagonally (the tails).  
A RIGHT arrow has a mirror-image pattern.  
HOG captures exactly this directional information.

**Feature vector size** for a 128×128 image with our parameters:  
`(128/8 - 1) × (128/8 - 1) × 2×2×9 = 15 × 15 × 36 = 8,100 values`

In [ ]:
# ── Find one example of each class ───────────────────────────────────────────
img_dir = os.path.join(ARCHIVE_DIR, "images", "train")
lbl_dir = os.path.join(ARCHIVE_DIR, "labels", "train")

examples = {}
for fname in sorted(os.listdir(img_dir)):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    info = read_label_file(fname, lbl_dir)
    if info and info[0] not in examples:
        examples[info[0]] = (fname, info)
    if len(examples) == 2:
        break

# ── Compute HOG with visualization ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
fig.suptitle("Step 2 — HOG Feature Visualization", fontsize=13, fontweight="bold")

for row, cls in [0, 1]:
    fname, info = examples[cls]
    img_array = preprocess(os.path.join(img_dir, fname), info)

    # visualize=True returns both the feature vector AND the HOG image
    features, hog_image = hog(img_array,
                               orientations=9,
                               pixels_per_cell=(8, 8),
                               cells_per_block=(2, 2),
                               visualize=True,
                               channel_axis=None)

    axes[row, 0].imshow(img_array, cmap="gray")
    axes[row, 0].set_title(f"Cropped grayscale — {CLASS_NAMES[cls]}", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(hog_image, cmap="magma")
    axes[row, 1].set_title(f"HOG visualization — {CLASS_NAMES[cls]}\n"
                            "(bright = strong gradient)", fontsize=9)
    axes[row, 1].axis("off")

    axes[row, 2].hist(features, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
    axes[row, 2].set_title(f"Feature value distribution — {CLASS_NAMES[cls]}", fontsize=9)
    axes[row, 2].set_xlabel("Feature value")
    axes[row, 2].set_ylabel("Count")

    print(f"Class {cls} ({CLASS_NAMES[cls]}) — {fname}")
    print(f"  Feature vector shape : {features.shape}")
    print(f"  Min / Max / Mean     : {features.min():.4f} / {features.max():.4f} / {features.mean():.4f}")
    print()

plt.tight_layout()
plt.show()
print("Notice: the HOG gradient patterns are MIRRORED between LEFT and RIGHT.")
print("This asymmetry is exactly what logistic regression will learn to separate.")

---
## Step 3 — Build the Feature Matrix X and Label Vector y

Now we apply the pipeline to **every image** in the train and test splits.

The result is two matrices:
- `X_train` — shape `(n_train_images, 8100)` — one HOG vector per row
- `y_train` — shape `(n_train_images,)` — 0 for LEFT, 1 for RIGHT

We save to `.npy` files so we don't have to reprocess images every time we retrain.

**Important:** If you already ran the `.py` scripts, the `.npy` files already exist and this cell will load them directly instead of reprocessing.

In [ ]:
def build_split(split_name):
    """Process all images in a split into (X, y) numpy arrays."""
    img_dir = os.path.join(ARCHIVE_DIR, "images", split_name)
    lbl_dir = os.path.join(ARCHIVE_DIR, "labels", split_name)
    X_list, y_list = [], []
    skipped = 0

    filenames = sorted(os.listdir(img_dir))
    for i, fname in enumerate(filenames):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        info = read_label_file(fname, lbl_dir)
        if info is None:
            skipped += 1
            continue
        arr  = preprocess(os.path.join(img_dir, fname), info)
        feat = hog(arr, **HOG_PARAMS)
        X_list.append(feat)
        y_list.append(info[0])
        if (i + 1) % 500 == 0:
            print(f"  [{split_name}] {len(X_list)} processed...")

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    print(f"  [{split_name}] Done. Skipped {skipped} (no label).")
    return X, y


# Load from disk if available, otherwise build from scratch
x_train_path = os.path.join(BASE_DIR, "X_train.npy")
x_test_path  = os.path.join(BASE_DIR, "X_test.npy")

if os.path.exists(x_train_path) and os.path.exists(x_test_path):
    print("Loading pre-built matrices from disk...")
    X_train = np.load(x_train_path)
    y_train = np.load(os.path.join(BASE_DIR, "y_train.npy"))
    X_test  = np.load(x_test_path)
    y_test  = np.load(os.path.join(BASE_DIR, "y_test.npy"))
else:
    print("Building matrices from images (this takes ~1 min)...")
    X_train, y_train = build_split("train")
    X_test,  y_test  = build_split("test")
    np.save(os.path.join(BASE_DIR, "X_train.npy"), X_train)
    np.save(os.path.join(BASE_DIR, "y_train.npy"), y_train)
    np.save(os.path.join(BASE_DIR, "X_test.npy"),  X_test)
    np.save(os.path.join(BASE_DIR, "y_test.npy"),  y_test)
    print("Saved to disk.")

print(f"\nX_train : {X_train.shape}  (samples × features)")
print(f"y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")

for split_name, y in [("train", y_train), ("test", y_test)]:
    u, c = np.unique(y, return_counts=True)
    print(f"\n{split_name} class distribution:")
    for cls, cnt in zip(u, c):
        print(f"  Class {cls} ({CLASS_NAMES[cls]}): {cnt} ({cnt/len(y)*100:.1f}%)")

---
## Step 4 — Train Logistic Regression

### How Logistic Regression Works

Logistic regression learns a **weight** for each of the 8,100 HOG features.  
It computes a weighted sum and passes it through the **sigmoid function**:

$$P(\text{RIGHT} \mid x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

- Output is always between 0 and 1 → interpreted as a probability
- Weights with large positive values push toward RIGHT
- Weights with large negative values push toward LEFT

### Why Standardize?

HOG features are already in [0,1] but have different variances.  
Standardizing (subtract mean, divide by std) ensures all features contribute equally during gradient descent.  
**Critical rule:** the scaler is fit on training data only — never on test data.

In [ ]:
# ── Standardize ───────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print("After scaling:")
print(f"  Train feature mean : {X_train_scaled.mean():.6f}  (should be ~0)")
print(f"  Train feature std  : {X_train_scaled.std():.6f}  (should be ~1)")

# ── Train ─────────────────────────────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
print(f"\nTrain accuracy: {train_acc*100:.2f}%")

print("\nFull classification report on TRAIN set:")
print(classification_report(y_train, model.predict(X_train_scaled),
                             target_names=["LEFT (0)", "RIGHT (1)"]))

# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(model,  os.path.join(BASE_DIR, "model.pkl"))
joblib.dump(scaler, os.path.join(BASE_DIR, "scaler.pkl"))
print("Saved model.pkl and scaler.pkl")

In [ ]:
# ── Visualize the learned coefficients ────────────────────────────────────────
# Each coefficient corresponds to one HOG feature.
# Positive coeff → that feature pushes toward RIGHT
# Negative coeff → that feature pushes toward LEFT

coef = model.coef_[0]
n_top = 15
top_idx    = np.argsort(np.abs(coef))[::-1][:n_top]
top_values = coef[top_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Step 4 — Logistic Regression Coefficients", fontsize=12, fontweight="bold")

axes[0].hist(coef, bins=60, color="steelblue", edgecolor="white", linewidth=0.2)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_title("All 8,100 coefficients")
axes[0].set_xlabel("Coefficient value")
axes[0].set_ylabel("Count")
axes[0].annotate("← pushes LEFT", xy=(-0.04, axes[0].get_ylim()[1]*0.9), fontsize=8, color="tomato")
axes[0].annotate("pushes RIGHT →", xy=(0.01, axes[0].get_ylim()[1]*0.9), fontsize=8, color="steelblue")

colors = ["tomato" if v > 0 else "steelblue" for v in top_values[::-1]]
axes[1].barh(range(n_top), top_values[::-1], color=colors)
axes[1].set_yticks(range(n_top))
axes[1].set_yticklabels([f"feat[{i}]" for i in top_idx[::-1]], fontsize=8)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Top {n_top} most influential features\n(red=RIGHT, blue=LEFT)")
axes[1].set_xlabel("Coefficient value")

plt.tight_layout()
plt.show()

---
## Step 5 — Probability Scores on the Test Set

### predict vs predict_proba

| Method | Output | What it does |
|--------|--------|--------------|
| `model.predict(X)` | `[0, 1, 0, ...]` | Applies threshold 0.5, returns class labels |
| `model.predict_proba(X)` | `[[0.3, 0.7], ...]` | Returns `[P(LEFT), P(RIGHT)]` for each sample |

For ROC analysis we always use **`predict_proba`**, specifically `[:, 1]` (the probability of being RIGHT).  
This gives a continuous score from 0 to 1 for each image — the raw signal we sweep thresholds over.

### What to Look For
- TRUE LEFT samples should cluster near **P(RIGHT) ≈ 0** (model says: not right)
- TRUE RIGHT samples should cluster near **P(RIGHT) ≈ 1** (model says: yes, right)
- Overlap in the middle = ambiguous zone, uncertainty

In [ ]:
# ── Get probability scores ────────────────────────────────────────────────────
proba_all   = model.predict_proba(X_test_scaled)   # shape: (402, 2)
proba_right = proba_all[:, 1]                        # P(RIGHT | x) for each test image

left_probs  = proba_right[y_test == 0]
right_probs = proba_right[y_test == 1]

print("P(RIGHT) statistics on test set:")
print(f"  True LEFT  — mean={left_probs.mean():.4f}  std={left_probs.std():.4f}")
print(f"  True RIGHT — mean={right_probs.mean():.4f}  std={right_probs.std():.4f}")
print()
print("Sample predictions (first 15 test images):")
print(f"  {'#':>4}  {'True':>8}  {'P(RIGHT)':>10}  {'Verdict'}")
print("  " + "-"*38)
for i in range(15):
    true  = CLASS_NAMES[y_test[i]]
    prob  = proba_right[i]
    pred  = "RIGHT" if prob >= 0.5 else "LEFT"
    ok    = "✓" if pred == true else "✗"
    print(f"  {i:>4}  {true:>8}  {prob:>10.4f}  {ok}")

In [ ]:
# ── Plot probability distributions ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Step 5 — Distribution of P(RIGHT | x) on the test set",
             fontsize=12, fontweight="bold")

# Overlapping histogram
axes[0].hist(left_probs,  bins=30, alpha=0.6, color="steelblue",
             label="True LEFT (0)",  density=True)
axes[0].hist(right_probs, bins=30, alpha=0.6, color="tomato",
             label="True RIGHT (1)", density=True)
axes[0].axvline(0.5, color="black", linestyle="--", linewidth=1.2, label="threshold = 0.5")
axes[0].set_xlabel("P(RIGHT | x)")
axes[0].set_ylabel("Density")
axes[0].set_title("Overlapping probability distributions")
axes[0].legend()

# Boxplot
bp = axes[1].boxplot([left_probs, right_probs],
                     tick_labels=["True LEFT", "True RIGHT"],
                     patch_artist=True)
bp["boxes"][0].set_facecolor("steelblue")
bp["boxes"][0].set_alpha(0.5)
bp["boxes"][1].set_facecolor("tomato")
bp["boxes"][1].set_alpha(0.5)
axes[1].axhline(0.5, color="black", linestyle="--", linewidth=1, label="threshold = 0.5")
axes[1].set_ylabel("P(RIGHT | x)")
axes[1].set_title("Boxplot per class")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Ideal: two non-overlapping distributions. Overlap = uncertain zone.")

---
## Step 6 — ROC Curve and AUC

### What is the ROC Curve?

We sweep the **classification threshold** from 0 → 1.  
At each threshold value t, every sample with `P(RIGHT) > t` is predicted RIGHT, the rest LEFT.  
We compute two rates:

$$\text{TPR} = \frac{TP}{TP + FN} \quad \text{(Recall — of all true RIGHTs, how many did we catch?)}$$

$$\text{FPR} = \frac{FP}{FP + TN} \quad \text{(of all true LEFTs, how many did we wrongly call RIGHT?)}$$

Plotting (FPR, TPR) for all thresholds gives the ROC curve.

### What is AUC?

AUC = Area Under the ROC Curve.  
**Intuition:** probability that a randomly chosen RIGHT sample scores higher than a randomly chosen LEFT sample.

| AUC | Meaning |
|-----|---------|
| 1.0 | Perfect classifier |
| 0.5 | Random guessing |
| < 0.5 | Worse than random (flip predictions) |

In [ ]:
# ── Compute ROC curve ─────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, proba_right, pos_label=1)
auc = roc_auc_score(y_test, proba_right)

print(f"AUC = {auc:.4f}")
print(f"Number of threshold points: {len(thresholds)}")
print()
print("What sklearn's roc_curve returns:")
print("  fpr        : false positive rates at each threshold (length N)")
print("  tpr        : true positive rates at each threshold  (length N)")
print("  thresholds : probability cutoffs used (length N-1, decreasing)")
print()
print("A few points on the curve:")
print(f"  {'FPR':>8}  {'TPR':>8}")
for f, t in zip(fpr, tpr):
    print(f"  {f:>8.4f}  {t:>8.4f}")

In [ ]:
# ── Find the point on the curve closest to the ideal corner (0,1) ─────────────
distances = np.sqrt(fpr**2 + (1 - tpr)**2)
best_idx  = np.argmin(distances)
best_fpr_corner  = fpr[best_idx]
best_tpr_corner  = tpr[best_idx]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Step 6 — ROC Curve", fontsize=13, fontweight="bold")

# Left: ROC curve
ax = axes[0]
ax.plot(fpr, tpr, color="darkorange", linewidth=2.5,
        label=f"ROC curve  (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], color="navy", linewidth=1.2, linestyle="--",
        label="Random baseline (AUC = 0.50)")
ax.scatter(best_fpr_corner, best_tpr_corner, color="red", zorder=5, s=100,
           label=f"Closest to (0,1)\n(FPR={best_fpr_corner:.2f}, TPR={best_tpr_corner:.2f})")
ax.fill_between(fpr, tpr, alpha=0.1, color="darkorange")  # shade the AUC area
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate (FPR)", fontsize=11)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=11)
ax.set_title("ROC Curve")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
ax.annotate("Perfect\ncorner", xy=(0, 1), xytext=(0.1, 0.85),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=8, color="gray")

# Right: TPR and FPR as a function of threshold
ax2 = axes[1]
t   = thresholds
tpr_t = tpr[1:] if len(tpr) > len(t) else tpr[:len(t)]
fpr_t = fpr[1:] if len(fpr) > len(t) else fpr[:len(t)]
ax2.plot(t, tpr_t, color="tomato",    linewidth=2, label="TPR (Recall/Sensitivity)")
ax2.plot(t, fpr_t, color="steelblue", linewidth=2, label="FPR (1 - Specificity)")
ax2.set_xlabel("Decision Threshold", fontsize=11)
ax2.set_ylabel("Rate", fontsize=11)
ax2.set_title("TPR and FPR vs Threshold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")
if auc >= 0.9:
    print("  Excellent — the model separates the two classes very well.")
elif auc >= 0.8:
    print("  Good.")
elif auc >= 0.7:
    print("  Acceptable.")
else:
    print("  Weak — consider revisiting feature extraction.")

---
## Step 7 — Choosing the Best Threshold (Youden Index)

### The Trade-off

The ROC curve shows that by moving the threshold we can trade FPR against TPR:  
- **Low threshold** → classify almost everything as RIGHT → high TPR but high FPR  
- **High threshold** → very conservative → low FPR but miss many true RIGHTs

The **Youden Index** selects the threshold that maximizes the total benefit:

$$J(t) = \text{TPR}(t) - \text{FPR}(t)$$

This is equivalent to finding the point on the ROC curve that is **farthest from the random diagonal**, measured vertically.  
It gives equal weight to both types of error — appropriate when misclassifying LEFT as RIGHT is equally bad as misclassifying RIGHT as LEFT.

### Final Evaluation

After choosing the threshold, we evaluate with a **confusion matrix**:

```
                  Predicted LEFT    Predicted RIGHT
  Actual LEFT         TN                FP
  Actual RIGHT        FN                TP
```

In [ ]:
# ── Youden Index ──────────────────────────────────────────────────────────────
# Align lengths (sklearn returns len(thresholds) = len(fpr) - 1)
tpr_t = tpr[1:] if len(tpr) > len(thresholds) else tpr[:len(thresholds)]
fpr_t = fpr[1:] if len(fpr) > len(thresholds) else fpr[:len(thresholds)]

J        = tpr_t - fpr_t
best_idx = np.argmax(J)

# Guard against edge case when AUC=1 and Youden picks boundary
best_threshold = thresholds[best_idx]
if best_threshold >= 1.0 or best_threshold <= 0.0:
    # Fall back to the point closest to (0,1) corner
    distances  = np.sqrt(fpr_t**2 + (1 - tpr_t)**2)
    best_idx   = np.argmin(distances)
    best_threshold = thresholds[best_idx]
    print(f"Note: Youden gave boundary value — using closest-to-(0,1) threshold instead.")

print(f"Youden Index results:")
print(f"  Best threshold  : {best_threshold:.4f}  (default = 0.5)")
print(f"  TPR at best     : {tpr_t[best_idx]:.4f}")
print(f"  FPR at best     : {fpr_t[best_idx]:.4f}")
print(f"  Specificity     : {1 - fpr_t[best_idx]:.4f}  (= 1 - FPR)")
print(f"  Youden J        : {J[best_idx]:.4f}")

In [ ]:
# ── Apply both thresholds and compare ────────────────────────────────────────
y_pred_default = (proba_right >= 0.5).astype(int)
y_pred_youden  = (proba_right >= best_threshold).astype(int)

cm_default = confusion_matrix(y_test, y_pred_default)
cm_youden  = confusion_matrix(y_test, y_pred_youden)

print("=" * 55)
print("EVALUATION — DEFAULT THRESHOLD (0.5)")
print("=" * 55)
print(classification_report(y_test, y_pred_default,
                             target_names=["LEFT (0)", "RIGHT (1)"]))

print("=" * 55)
print(f"EVALUATION — YOUDEN THRESHOLD ({best_threshold:.3f})")
print("=" * 55)
print(classification_report(y_test, y_pred_youden,
                             target_names=["LEFT (0)", "RIGHT (1)"]))

In [ ]:
# ── Final plot: ROC + both confusion matrices ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Step 7 — Threshold Selection & Final Evaluation",
             fontsize=13, fontweight="bold")

# ROC with both thresholds marked
ax = axes[0]
ax.plot(fpr, tpr, color="darkorange", linewidth=2,
        label=f"ROC (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")

# Youden threshold
ax.scatter(fpr_t[best_idx], tpr_t[best_idx], color="red", zorder=5, s=120,
           label=f"Youden  thresh={best_threshold:.2f}\n"
                 f"TPR={tpr_t[best_idx]:.2f}  FPR={fpr_t[best_idx]:.2f}")

# Default 0.5 threshold — find nearest point
diff_05  = np.abs(thresholds - 0.5)
idx_05   = np.argmin(diff_05)
ax.scatter(fpr_t[idx_05], tpr_t[idx_05], color="blue", zorder=5, s=80, marker="s",
           label=f"Default  thresh=0.50\n"
                 f"TPR={tpr_t[idx_05]:.2f}  FPR={fpr_t[idx_05]:.2f}")

ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC — Both Thresholds")
ax.legend(fontsize=7, loc="lower right")
ax.grid(alpha=0.3)

# Confusion matrix — default
ConfusionMatrixDisplay(cm_default, display_labels=["LEFT", "RIGHT"]).plot(
    ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Confusion Matrix\nDefault threshold = 0.5")

# Confusion matrix — Youden
ConfusionMatrixDisplay(cm_youden, display_labels=["LEFT", "RIGHT"]).plot(
    ax=axes[2], colorbar=False, cmap="Oranges")
axes[2].set_title(f"Confusion Matrix\nYouden threshold = {best_threshold:.3f}")

plt.tight_layout()
plt.show()

# Summary table
tn0, fp0, fn0, tp0 = cm_default.ravel()
tn1, fp1, fn1, tp1 = cm_youden.ravel()
print(f"\n{'':35s}  {'Default (0.5)':>14}  {'Youden':>8}")
print(f"{'True Positives  (RIGHT → RIGHT)':35s}  {tp0:>14}  {tp1:>8}")
print(f"{'False Positives (LEFT  → RIGHT)':35s}  {fp0:>14}  {fp1:>8}")
print(f"{'True Negatives  (LEFT  → LEFT)':35s}  {tn0:>14}  {tn1:>8}")
print(f"{'False Negatives (RIGHT → LEFT)':35s}  {fn0:>14}  {fn1:>8}")
print(f"{'Accuracy':35s}  {(tp0+tn0)/(tp0+tn0+fp0+fn0)*100:>13.1f}%  {(tp1+tn1)/(tp1+tn1+fp1+fn1)*100:>7.1f}%")

---
## Using the Trained Model on New Images

The model is saved to disk as two files:  
- `model.pkl` — the logistic regression weights  
- `scaler.pkl` — the StandardScaler (always needed alongside the model)

Use this function to predict any new image:

In [ ]:
def predict_image(img_path, label_path=None):
    """
    Predict LEFT or RIGHT for a single image.

    img_path   : path to any .jpg or .png image
    label_path : optional YOLO .txt file for bounding box crop
                 (if not provided, uses the full image)
    """
    loaded_model  = joblib.load(os.path.join(BASE_DIR, "model.pkl"))
    loaded_scaler = joblib.load(os.path.join(BASE_DIR, "scaler.pkl"))

    # Try to get bounding box
    bbox = None
    if label_path and os.path.exists(label_path):
        with open(label_path) as f:
            parts = f.readline().strip().split()
        if len(parts) >= 5:
            bbox = (0, float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]))

    # Preprocess
    if bbox:
        arr = preprocess(img_path, bbox)
        note = "with bbox crop"
    else:
        img = Image.open(img_path).convert("L").resize(IMG_SIZE, Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        note = "full image (no bbox)"

    # Feature extraction
    feat = hog(arr, **HOG_PARAMS).reshape(1, -1)
    feat = loaded_scaler.transform(feat)

    # Predict
    proba = loaded_model.predict_proba(feat)[0]
    pred  = int(loaded_model.predict(feat)[0])

    print(f"Image      : {img_path}  [{note}]")
    print(f"Prediction : {CLASS_NAMES[pred]}  (class {pred})")
    print(f"Confidence : {proba[pred]*100:.1f}%")
    print(f"  P(LEFT)  = {proba[0]:.4f}")
    print(f"  P(RIGHT) = {proba[1]:.4f}")
    return CLASS_NAMES[pred], proba[pred]


# ── Example: predict a single test image ─────────────────────────────────────
sample_img = os.path.join(ARCHIVE_DIR, "images", "test", "1641.jpg")
sample_lbl = os.path.join(ARCHIVE_DIR, "labels", "test", "1641.txt")

predict_image(sample_img, sample_lbl)

---
## Summary

### What We Built

```
Raw image (640×480 RGB)
        ↓
Crop to bounding box  ← uses YOLO label coordinates
        ↓
Convert to grayscale
        ↓
Resize to 128×128
        ↓
HOG feature extraction  →  vector of 8,100 numbers
        ↓
StandardScaler  →  zero mean, unit variance
        ↓
Logistic Regression  →  P(RIGHT | x) ∈ [0, 1]
        ↓
Sweep threshold  →  ROC curve  →  AUC
        ↓
Youden Index  →  optimal threshold  →  final predictions
```

### Key Results

| Metric | Value |
|--------|-------|
| Train samples | 2,645 (47% LEFT, 53% RIGHT) |
| Test samples | 402 (50% / 50%) |
| HOG features per image | 8,100 |
| Train accuracy | 100% |
| AUC | 1.000 |
| Test accuracy (threshold=0.5) | 100% |

### Why Did This Work So Well?

1. **Bounding box crop** removed the background — HOG only saw the arrow
2. **HOG** captured the exact geometric difference between left-pointing and right-pointing edges
3. The arrow shapes are **visually very distinct** — this is a well-posed binary classification problem

### Key Concept Recap: ROC vs Accuracy

- **Accuracy** tells you how many predictions are correct at a fixed threshold (0.5)
- **ROC curve** shows the full range of trade-offs between catching true positives (TPR) and avoiding false alarms (FPR)
- **AUC** summarizes the entire curve in one number — it is independent of any threshold
- **Youden Index** gives you a principled way to choose the operating threshold after training